# 05 — LSTM mínima (dirección del precio)

Pipeline básico con **PyTorch**: ventanas temporales sobre features de mercado, misma etiqueta `label` que en notebooks anteriores (1 = precio sube al día siguiente).

- **Varios mercados:** se leen todos los `prices_*.csv` (o la lista que definas); el split temporal es **por mercado**; las ventanas LSTM **no cruzan** un mercado con otro.
- **`StandardScaler` ajustado solo en train** (pool de todos los mercados) antes de formar secuencias.
- Secuencias **solo dentro** del tramo train o test de cada mercado (sin mezclar cortes).

**Entorno:** instala dependencias en el venv del repo (`pip install -r requirements.txt`), actívalo con `.\.venv\Scripts\Activate.ps1` y en Jupyter elige el kernel **Python 3 (predikt)** (registrado con ese intérprete). Así `torch` y el resto coinciden con el proyecto.

**Rutas:** ejecutar con el servidor Jupyter lanzado **desde el venv**, o abrir el notebook con `cwd` en `notebooks/` para que `ROOT` resuelva bien al repo.

In [11]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from src.features import build_market_features, build_labels, temporal_split

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Configuración

- **`MARKET_FILES`**: por defecto **todos** los `data/raw/prices_*.csv`; puedes sustituir por una lista de rutas. **`MAX_MARKETS`**: recorta a los N primeros (orden alfabético) si no quieres cargar cientos de archivos.
- El **split 70/30 es por mercado** (en el tiempo: primeros días de esa serie → train, últimos → test); al entrenar se **mezclan secuencias** de todos los mercados, pero cada ventana solo usa días de **un** `slug`.

In [13]:
RAW_DIR = ROOT / "data" / "raw"
# Todos los mercados con precios diarios:
MARKET_FILES = sorted(RAW_DIR.glob("prices_*.csv"))
# O una lista explícita (comenta las dos líneas anteriores y descomenta):
# MARKET_FILES = [RAW_DIR / "prices_2023-24-detroit-pistons-worst-nba-team-of-all-time.csv"]

# None = cargar todos. Si tienes muchos CSV, pon un número para ir más rápido (ej. 50).
MAX_MARKETS = None
if MAX_MARKETS is not None:
    MARKET_FILES = MARKET_FILES[:MAX_MARKETS]

GROUP_COL = "slug"  # columna que identifica el mercado (no entra como feature)

LOOKBACK = 14          # días por ventana
TRAIN_RATIO = 0.70
EPOCHS = 80
BATCH_SIZE = 16
HIDDEN_SIZE = 32
LSTM_LAYERS = 1
LEARNING_RATE = 1e-3

SAVE_MODEL = True
MODEL_PATH = PROC_DIR / "lstm_baseline.pt"

## 1. Cargar datos y construir features

In [14]:
def load_one_market(path: Path) -> pd.DataFrame:
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["date"], utc=True)
    d = d.sort_values("date").reset_index(drop=True)
    d = build_market_features(d, price_col="price")
    d = build_labels(d, price_col="price")
    d = d.dropna().reset_index(drop=True)
    d = d.dropna(subset=["label"]).reset_index(drop=True)
    d["label"] = d["label"].astype(np.float64)
    return d


if not MARKET_FILES:
    raise FileNotFoundError(
        f"No hay CSV en {RAW_DIR}/prices_*.csv. Descarga datos o ajusta MARKET_FILES."
    )

parts = [load_one_market(p) for p in MARKET_FILES]
df = pd.concat(parts, ignore_index=True)

EXCLUDE = {"date", "slug", "question", "label"}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]

n_m = df[GROUP_COL].nunique()
print(f"Archivos: {len(MARKET_FILES)}, mercados (slugs): {n_m}, filas totales: {len(df)}")
df.head()

Archivos: 500, mercados (slugs): 427, filas totales: 27250


,date,price,slug,question,ret_1d,ret_3d,ret_7d,ma7,ma14,ma_ratio,vol7,label
0,2024-01-05 00:00:00+00:00,0.15,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.105361,0.265703,-0.448950,0.145714,0.156875,0.928856,0.159012,0.0
1,2024-01-06 00:00:00+00:00,0.15,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.000000,0.105361,-0.287682,0.138571,0.156111,0.887646,0.154204,0.0
2,2024-01-07 00:00:00+00:00,0.12,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,-0.223144,-0.117783,-0.287682,0.132857,0.152500,0.871194,0.154204,0.0
3,2024-01-08 00:00:00+00:00,0.12,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.000000,-0.223144,-0.040822,0.132143,0.149545,0.883630,0.124712,1.0
4,2024-01-09 00:00:00+00:00,0.13,2023-24-detroit-pistons-worst-nba-team-of-all-...,2023-24 Detroit Pistons worst NBA team of all ...,0.080043,-0.143101,0.122602,0.134286,0.147917,0.907847,0.123060,0.0


## 2. Split temporal **por mercado** y escalado (solo train)

Para cada `slug`, el 70 % inicial (en el tiempo) va a train y el 30 % final a test. El escalador se ajusta con **todas** las filas de train de todos los mercados.

In [15]:
train_parts, test_parts = [], []
for _, g in df.groupby(GROUP_COL):
    g = g.sort_values("date").reset_index(drop=True)
    tr, te = temporal_split(g, train_ratio=TRAIN_RATIO, date_col="date")
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].values)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [16]:
def make_sequences(features: np.ndarray, labels: np.ndarray, lookback: int):
    """features: (n, n_feat), labels: (n,) — etiqueta al último día de la ventana."""
    X_list, y_list = [], []
    for t in range(lookback - 1, len(features)):
        X_list.append(features[t - lookback + 1 : t + 1])
        y_list.append(labels[t])
    return np.stack(X_list, axis=0), np.array(y_list, dtype=np.float32)


def sequences_from_grouped(split_df: pd.DataFrame, lookback: int):
    """Una ventana LSTM nunca cruza dos mercados distintos."""
    X_blocks, y_blocks = [], []
    for _, g in split_df.groupby(GROUP_COL):
        g = g.sort_values("date")
        if len(g) < lookback:
            continue
        Xs = scaler.transform(g[FEATURE_COLS].values)
        ys = g["label"].values.astype(np.float32)
        X_seq, y_seq = make_sequences(Xs, ys, lookback)
        if len(y_seq) == 0:
            continue
        X_blocks.append(X_seq)
        y_blocks.append(y_seq)
    if not X_blocks:
        return np.empty((0, lookback, len(FEATURE_COLS))), np.array([], dtype=np.float32)
    return np.concatenate(X_blocks, axis=0), np.concatenate(y_blocks, axis=0)


X_train, y_train = sequences_from_grouped(train_df, LOOKBACK)
X_test, y_test = sequences_from_grouped(test_df, LOOKBACK)

print(f"Train secuencias: {len(y_train)}, Test secuencias: {len(y_test)}")
if len(y_train) == 0 or len(y_test) == 0:
    raise ValueError(
        "Muy pocas filas por mercado o LOOKBACK demasiado grande. "
        "Añade más CSV, reduce LOOKBACK o revisa mercados cortos."
    )

Train secuencias: 13951, Test secuencias: 4015


## 3. DataLoaders

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).float(),
)

train_loader = DataLoader(
    train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True, drop_last=False
)
test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

## 4. Modelo: LSTM + capa densa

Salida **sigmoid** = probabilidad estimada de subida.

In [18]:
n_features = X_train.shape[2]


class LSTMClassifier(nn.Module):
    def __init__(self, n_feat: int, hidden: int, n_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_feat,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(-1)


model = LSTMClassifier(n_features, HIDDEN_SIZE, LSTM_LAYERS).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

LSTMClassifier(
  (lstm): LSTM(8, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

## 5. Entrenamiento

In [19]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg = total_loss / len(train_ds)
    if epoch == 1 or epoch % max(1, EPOCHS // 10) == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:4d}  loss {avg:.4f}")

epoch    1  loss 0.5013
epoch    8  loss 0.4769
epoch   16  loss 0.4686
epoch   24  loss 0.4582
epoch   32  loss 0.4442
epoch   40  loss 0.4303
epoch   48  loss 0.4123
epoch   56  loss 0.3966
epoch   64  loss 0.3802
epoch   72  loss 0.3656
epoch   80  loss 0.3487


## 6. Evaluación en test

Umbral 0.5 en **probabilidad** (`sigmoid(logits)`).

In [20]:
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        y_true = yb.numpy()

y_pred = (probs >= 0.5).astype(int)
y_true_int = y_true.astype(int)

acc = accuracy_score(y_true_int, y_pred)
f1 = f1_score(y_true_int, y_pred, zero_division=0)

print(f"Accuracy: {acc:.3f}")
print(f"F1 (binary): {f1:.3f}")

if SAVE_MODEL:
    torch.save(
        {
            "model_state": model.state_dict(),
            "feature_cols": FEATURE_COLS,
            "lookback": LOOKBACK,
            "hidden_size": HIDDEN_SIZE,
            "n_layers": LSTM_LAYERS,
            "scaler_mean": scaler.mean_,
            "scaler_scale": scaler.scale_,
            "group_col": GROUP_COL,
            "n_markets": int(df[GROUP_COL].nunique()),
            "csv_stems": [p.stem for p in MARKET_FILES],
        },
        MODEL_PATH,
    )
    print(f"Guardado: {MODEL_PATH}")

Accuracy: 0.716
F1 (binary): 0.265
Guardado: c:\Users\oscar\repos\predikt\data\processed\lstm_baseline.pt
